# External held-out — three approaches on JailbreakBench

100 prompts from [**JailbreakBench**](https://huggingface.co/datasets/JailbreakBench/JBB-Behaviors) (Chao et al., 2024) — first 50 harmful + 50 behavior-matched benign, translated to Polish with Claude and reviewed manually (English originals in `original_en`).

Three classifiers on the same disjoint test set:
1. **SetFit** — our pipeline, fine-tuned on 80% of in-house dataset
2. **NLI zero-shot** — `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` with Polish hypotheses
3. **ProtectAI** — `ProtectAI/deberta-v3-base-prompt-injection-v2` (English-trained baseline)

None of the three has seen these 100 prompts during training.


In [1]:
import json
import pandas as pd
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from tqdm import tqdm

rows = [json.loads(line) for line in open('../data/external.jsonl', encoding='utf-8') if line.strip()]
df = pd.DataFrame(rows)
df['true_unsafe'] = df['label'].eq('unsafe')
print(df['label'].value_counts())


label
unsafe    50
safe      50
Name: count, dtype: int64


## 1. SetFit (ours)

In [2]:
import yaml
from setfit import SetFitModel

threshold = yaml.safe_load(open('../params.yaml'))['threshold']
sf = SetFitModel.from_pretrained('../models/setfit')
proba = sf.predict_proba(df['prompt'].tolist())
df['sf_unsafe'] = [float(p[1]) >= threshold for p in proba.tolist()]
print(f'threshold = {threshold}')
print(classification_report(df['true_unsafe'], df['sf_unsafe'], target_names=['safe', 'unsafe'], digits=3))


threshold = 0.3
              precision    recall  f1-score   support

        safe      0.897     0.520     0.658        50
      unsafe      0.662     0.940     0.777        50

    accuracy                          0.730       100
   macro avg      0.779     0.730     0.718       100
weighted avg      0.779     0.730     0.718       100



## 2. NLI zero-shot

In [3]:
from transformers import pipeline

nli = pipeline('zero-shot-classification', model='MoritzLaurer/mDeBERTa-v3-base-mnli-xnli')
LABELS = {'neutralne pytanie lub prośba': 'safe',
          'szkodliwa, nielegalna lub niebezpieczna prośba': 'unsafe'}

pred = []
for _, r in tqdm(df.iterrows(), total=len(df), desc='NLI'):
    out = nli(r['prompt'], candidate_labels=list(LABELS.keys()), multi_label=False)
    pred.append(LABELS[out['labels'][0]] == 'unsafe')
df['nli_unsafe'] = pred
print(classification_report(df['true_unsafe'], df['nli_unsafe'], target_names=['safe', 'unsafe'], digits=3))


/Users/michal/Library/Code/PROJEKTY/guard-pl/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
NLI: 100%|██████████| 100/100 [00:12<00:00,  8.16it/s]

              precision    recall  f1-score   support

        safe      0.632     0.480     0.545        50
      unsafe      0.581     0.720     0.643        50

    accuracy                          0.600       100
   macro avg      0.606     0.600     0.594       100
weighted avg      0.606     0.600     0.594       100



## 3. ProtectAI

In [4]:
pa = pipeline('text-classification', model='ProtectAI/deberta-v3-base-prompt-injection-v2', truncation=True, max_length=512)

pred = []
for _, r in tqdm(df.iterrows(), total=len(df), desc='ProtectAI'):
    pred.append(pa(r['prompt'])[0]['label'] == 'INJECTION')
df['pa_unsafe'] = pred
print(classification_report(df['true_unsafe'], df['pa_unsafe'], target_names=['safe', 'unsafe'], digits=3))


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
ProtectAI: 100%|██████████| 100/100 [00:07<00:00, 13.72it/s]

              precision    recall  f1-score   support

        safe      0.500     0.720     0.590        50
      unsafe      0.500     0.280     0.359        50

    accuracy                          0.500       100
   macro avg      0.500     0.500     0.475       100
weighted avg      0.500     0.500     0.475       100



## Comparison

In [5]:
def m(yp, yt):
    return {'precision': precision_score(yt, yp, zero_division=0),
            'recall': recall_score(yt, yp, zero_division=0),
            'f1': f1_score(yt, yp, zero_division=0),
            'accuracy': (yt == yp).mean()}

yt = df['true_unsafe']
pd.DataFrame({
    'SetFit (ours)': m(df['sf_unsafe'], yt),
    'NLI zero-shot': m(df['nli_unsafe'], yt),
    'ProtectAI': m(df['pa_unsafe'], yt),
}).T.round(3).sort_values('f1', ascending=False)


,precision,recall,f1,accuracy
SetFit (ours),0.662,0.94,0.777,0.73
NLI zero-shot,0.581,0.72,0.643,0.60
ProtectAI,0.500,0.28,0.359,0.50


## Per-category

In [6]:
for cat in sorted(df['category'].unique()):
    sub = df[df['category'] == cat]
    yt = sub['true_unsafe']
    sf = (sub['sf_unsafe'] == yt).sum()
    nli = (sub['nli_unsafe'] == yt).sum()
    pa = (sub['pa_unsafe'] == yt).sum()
    print(f'{cat:30}  SetFit {sf:2}/{len(sub)}  NLI {nli:2}/{len(sub)}  ProtectAI {pa:2}/{len(sub)}')


Economic harm                   SetFit 15/20  NLI 10/20  ProtectAI 11/20
Fraud/Deception                 SetFit 15/20  NLI 12/20  ProtectAI  9/20
Harassment/Discrimination       SetFit 14/20  NLI 12/20  ProtectAI 10/20
Malware/Hacking                 SetFit 12/20  NLI 14/20  ProtectAI 10/20
Physical harm                   SetFit 17/20  NLI 12/20  ProtectAI 10/20


## Notes

SetFit beats both zero-shot baselines and an English-trained guard. Contrastive fine-tuning on 220 Polish prompts is enough to learn the safe/unsafe boundary across behavior-matched pairs that confuse pure embedding similarity.
